# Práctica 4: Planificación automática
# Inteligencia Artificial
# Grado en Ingeniería Informática - Ingeniería del Software
# Universidad de Sevilla

[PDDL](https://planning.wiki/) (_Planning Domain Definition Language_) es una familia de lenguajes que se ha convertido en el estándar para la especificación de dominios y problemas de planificación automática. A medida que este subcampo de la Inteligencia Artificial ha evolucionado, así lo ha hecho también el lenguaje usado para describir los problemas que estudia. Hoy en día hay, por tanto, varias versiones disponibles de PDDL, con diferentes niveles de expresividad.

La [versión 1.2](https://planning.wiki/ref/pddl) de PDDL establece la sintaxis más básica que debe ser entendible por cualquier planificador y es a la que nos referiremos al hablar simplemente de PDDL y la que usaremos en esta práctica.

Un problema de planificación automática se especifica en PDDL en dos partes: por un lado, se especifica el dominio del problema, que establece todos aquellos aspectos (predicados y acciones) que son independientes de la situación concreta que se está tratando de resolver; por otro lado, se especifica el problema en sí, estableciendo exactamente qué objetos existen, en qué situación inicial se encuentran y a qué situación final se pretende llegar.

La sintaxis mínima para el fichero de especificación del dominio es la siguiente:

```
(define
  (domain <nombre_del_dominio>)
  (:requirements :strips)
  (:predicates
    (<nombre_del_predicado> <argumento_1> ... <argumento_m>)
    ...
  )
  (:action <nombre_de_la_acción>
    :parameters (<argumento_1> ... <argumento_n>)
    :precondition (and
      <precondición>
      ...
    )
    :effect (and
      (not <borrado>)
      ...
      <adición>
      ...
    )
  ...
  )
)
```

Por ejemplo, el dominio del mundo de los bloques se podría representar en un fichero `dominio_mundo_bloques.pddl` con el siguiente contenido:

```
(define
  (domain dominio_mundo_bloques)
  (:requirements :strips)
  (:predicates
    (sobre_la_mesa ?b)
    (sobre ?b1 ?b2)
    (agarrado ?b)
    (brazo_libre)
    (despejado ?b)
  )
  (:action agarrar
    :parameters (?b)
    :precondition (and
      (sobre_la_mesa ?b)
      (despejado ?b)
      (brazo_libre)
    )
    :effect (and
      (not (sobre_la_mesa ?b))
      (not (despejado ?b))
      (not (brazo_libre))
      (agarrado ?b)
    )
  )
  (:action bajar
    :parameters (?b)
    :precondition (and
      (agarrado ?b)
    )
    :effect (and
      (not (agarrado ?b))
      (sobre_la_mesa ?b)
      (despejado ?b)
      (brazo_libre)
    )
  )
  (:action desapilar
    :parameters (?b1 ?b2)
    :precondition (and
      (sobre ?b1 ?b2)
      (despejado ?b1)
      (brazo_libre)
    )
    :effect (and
      (not (sobre ?b1 ?b2))
      (not (despejado ?b1))
      (not (brazo_libre))
      (agarrado ?b1)
      (despejado ?b2)
    )
  )
  (:action apilar
    :parameters (?b1 ?b2)
    :precondition (and
      (agarrado ?b1)
      (despejado ?b2)
    )
    :effect (and
      (not (agarrado ?b1))
      (not (despejado ?b2))
      (sobre ?b1 ?b2)
      (despejado ?b1)
      (brazo_libre)
    )
  )
)
```

La sintaxis mínima para el fichero de especificación del problema es la siguiente:

```
(define
  (problem <nombre_del_problema>)
  (:domain <dominio_del_problema>)
  (:objects
    <nombre_de_objeto>
    ...
  )
  (:init
    <predicado>
    ...
  )
  (:goal (and
      <objetivo>
      ...
    )
  )
)
```

Por ejemplo, un problema del mundo de los bloques se podría representar en un fichero `problema_mundo_bloques.pddl` con el siguiente contenido:

```
(define
  (problem problema_mundo_bloques)
  (:domain dominio_mundo_bloques)
  (:objects
    A B C
  )
  (:init
    (sobre A C)
    (sobre_la_mesa B)
    (sobre_la_mesa C)
    (despejado A)
    (despejado B)
    (brazo_libre)
  )
  (:goal (and
      (sobre_la_mesa A)
      (sobre B A)
      (sobre C B)
    )
  )
)
```

En esta práctica se hará uso de la biblioteca de Python [`Unified-Planning`](https://unified-planning.readthedocs.io), que proporciona una capa de abstracción que permite especificar problemas de planificación de forma independiente y luego utilizar uno de los planificadores disponibles instalados en el sistema, que en nuestro caso será el planificador [`Fast Downward`](https://www.fast-downward.org).

Esta práctica requiere, por tanto, tener instalados los paquetes de Python `unified-planning` y `up-fast-downward`.

La biblioteca `Unified-Planning` provee de herramientas para leer y escribir dominios y problemas de planificación escritos en PDDL hasta la versión 2.1 nivel 3.

In [1]:
from unified_planning.io import PDDLReader

In [2]:
lector_PDDL = PDDLReader()
problema_mundo_bloques = lector_PDDL.parse_problem('dominio_mundo_bloques.pddl',
                                                   'problema_mundo_bloques.pddl')

En planificación automática es habitual trabajar con el concepto de _fluente_, que son funciones que describen propiedades de sus argumentos que pueden cambiar con la ejecución de las acciones del problema. Los _fluentes_ generalizan el concepto de predicado, ya que estos describen propiedades booleanas (es decir, con valor verdadero o falso únicamente), mientras que, por ejemplo, es posible considerar _fluentes_ que describan propiedades numéricas.

Por otra parte, la biblioteca `Unified-Planning` permite establecer valores por defecto para los _fluentes_. Esto facilita la implementación de la hipótesis del mundo cerrado, ya que basta establecer `falso` como valor por defecto para todos los predicados e indicar luego `verdadero` como valor únicamente para los hechos del estado inicial.

In [3]:
print(problema_mundo_bloques)

problem name = problema_mundo_bloques

types = [object]

fluents = [
  bool sobre_la_mesa[b=object]
  bool sobre[b1=object, b2=object]
  bool agarrado[b=object]
  bool brazo_libre
  bool despejado[b=object]
]

actions = [
  action agarrar(object b) {
    preconditions = [
      (sobre_la_mesa(b) and despejado(b) and brazo_libre)
    ]
    effects = [
      sobre_la_mesa(b) := false
      despejado(b) := false
      brazo_libre := false
      agarrado(b) := true
    ]
  }
  action bajar(object b) {
    preconditions = [
      agarrado(b)
    ]
    effects = [
      agarrado(b) := false
      sobre_la_mesa(b) := true
      despejado(b) := true
      brazo_libre := true
    ]
  }
  action desapilar(object b1, object b2) {
    preconditions = [
      (sobre(b1, b2) and despejado(b1) and brazo_libre)
    ]
    effects = [
      sobre(b1, b2) := false
      despejado(b1) := false
      brazo_libre := false
      agarrado(b1) := true
      despejado(b2) := true
    ]
  }
  action apilar(objec

La biblioteca `Unified-Planning` es compatible con varios modos de operación. Para encontrar un plan solución de un problema mediante la ejecución de un planificador hay que usar el modo `OneshotPlanner`.

In [4]:
from unified_planning.shortcuts import OneshotPlanner

Para cada problema de planificación, la biblioteca `Unified-Planning` calcula su _tipo_, que es una colección de indicadores que identifica las características utilizadas en la especificación de problema.

In [5]:
print(problema_mundo_bloques.kind)

PROBLEM_CLASS: ['ACTION_BASED']
TYPING: ['FLAT_TYPING']


Esto permite determinar si el planificador que se pretende utilizar para resolver el problema es compatible con la especificación de este último.

In [6]:
planificador = OneshotPlanner(name='fast-downward')
planificador.supports(problema_mundo_bloques.kind)

NOTE: To disable printing of planning engine credits, add this line to your code: `up.shortcuts.get_environment().credits_stream = None`
  *** Credits ***
  * In operation mode `OneshotPlanner` at line 1 of `/tmp/ipykernel_28051/400592943.py`, you are using the following planning engine:
  * Engine name: Fast Downward
  * Developers:  Uni Basel team and contributors (cf. https://github.com/aibasel/downward/blob/main/README.md)
  * Description: Fast Downward is a domain-independent classical planning system.



True

Puesto que el planificador `Fast Downward` es compatible con la especificación proporcionada del problema de los bloques, podemos usarlo para buscar un plan solución.

In [7]:
resultado = planificador.solve(problema_mundo_bloques)
print(resultado)

status: SOLVED_SATISFICING
engine: Fast Downward
plan: SequentialPlan:
    desapilar(a, c)
    bajar(a)
    agarrar(b)
    apilar(b, a)
    agarrar(c)
    apilar(c, b)


En lugar de leer la especificación de un problema de planificación automática a partir de los ficheros PDDL, la biblioteca `Unified-Planning` permite realizar esa especificación mediante código. Como ejemplo vamos a construir a continuación una especificación del problema de transporte de paquetes descrito en el tema.

En primer lugar, hay que crear los tipos de objetos que se usarán en la especificación, pudiéndose establecer una jerarquía entre los mismos. Para cada tipo de objeto hay que indicar su nombre y, en su caso, de qué tipo de objeto es subtipo.

In [8]:
from unified_planning.shortcuts import UserType

In [9]:
paquete = UserType('Paquete')
localización = UserType('Localizacion')
lugar = UserType('Lugar', father=localización)
camión = UserType('Camion', father=localización)

In [10]:
for tipo in [paquete, localización, lugar, camión]:
    print(tipo)

Paquete
Localizacion
Lugar - Localizacion
Camion - Localizacion


A continuación se crean los predicados como _fluentes_ booleanos. Para cada predicado hay que indicar su nombre y el tipo de objeto de cada uno de sus argumentos (obsérvese que establecemos que los camiones pueden estar en un lugar, mientras que los paquetes pueden estar en una localización, que engloba tanto a los lugares como a los camiones).

In [11]:
from unified_planning.shortcuts import Fluent, BoolType

In [12]:
conectados = Fluent('CONECTADOS', BoolType(), l1=lugar, l2=lugar)
camión_en = Fluent('CAMION_EN', BoolType(), c=camión, l=lugar)
paquete_en = Fluent('PAQUETE_EN', BoolType(), p=paquete, lc=localización)

In [13]:
for predicado in [conectados, camión_en, paquete_en]:
    print(predicado)

bool CONECTADOS[l1=Lugar - Localizacion, l2=Lugar - Localizacion]
bool CAMION_EN[c=Camion - Localizacion, l=Lugar - Localizacion]
bool PAQUETE_EN[p=Paquete, lc=Localizacion]


Finalmente, se crean las acciones. Para cada acción hay que indicar su nombre y el tipo de objeto de cada uno de sus argumentos y luego, uno a uno, añadir los hechos precondiciones, los hechos de la lista de borrado como efectos con valor falso y los hechos de la lista de adición como efectos con valor verdadero.

In [14]:
from unified_planning.shortcuts import InstantaneousAction

In [15]:
acción_ir = InstantaneousAction('IR', c=camión, l1=lugar, l2=lugar)
c = acción_ir.c
l1 = acción_ir.l1
l2 = acción_ir.l2
for hecho in [conectados(l1, l2),
              camión_en(c, l1)]:
    acción_ir.add_precondition(hecho)
for hecho in [camión_en(c, l1)]:
    acción_ir.add_effect(hecho, False)
for hecho in [camión_en(c, l2)]:
    acción_ir.add_effect(hecho, True)
print(acción_ir)

action IR(Camion - Localizacion c, Lugar - Localizacion l1, Lugar - Localizacion l2) {
    preconditions = [
      CONECTADOS(l1, l2)
      CAMION_EN(c, l1)
    ]
    effects = [
      CAMION_EN(c, l1) := false
      CAMION_EN(c, l2) := true
    ]
  }


Obsérvese cómo el referenciar los argumentos de la acción mediante las variables `c`, `l1` y `l2` facilita la escritura posterior de los hechos al incluir las precondiciones, la lista de borrado y la lista de adición. Sin esas referencias habría que escribir los hechos como `conectados(acción_ir.l1, acción_ir.l2)`, `camión_en(acción_ir.c, acción_ir.l1)`, etcétera.

In [16]:
acción_cargar = InstantaneousAction('CARGAR', p=paquete, c=camión, l=lugar)
p = acción_cargar.p
c = acción_cargar.c
l = acción_cargar.l
for hecho in [camión_en(c, l),
              paquete_en(p, l)]:
    acción_cargar.add_precondition(hecho)
for hecho in [paquete_en(p, l)]:
    acción_cargar.add_effect(hecho, False)
for hecho in [paquete_en(p, c)]:
    acción_cargar.add_effect(hecho, True)
print(acción_cargar)

action CARGAR(Paquete p, Camion - Localizacion c, Lugar - Localizacion l) {
    preconditions = [
      CAMION_EN(c, l)
      PAQUETE_EN(p, l)
    ]
    effects = [
      PAQUETE_EN(p, l) := false
      PAQUETE_EN(p, c) := true
    ]
  }


In [17]:
acción_descargar = InstantaneousAction('DESCARGAR', p=paquete, c=camión, l=lugar)
p = acción_descargar.p
c = acción_descargar.c
l = acción_descargar.l
for hecho in [camión_en(c, l),
              paquete_en(p, c)]:
    acción_descargar.add_precondition(hecho)
for hecho in [paquete_en(p, c)]:
    acción_descargar.add_effect(hecho, False)
for hecho in [paquete_en(p, l)]:
    acción_descargar.add_effect(hecho, True)
print(acción_descargar)

action DESCARGAR(Paquete p, Camion - Localizacion c, Lugar - Localizacion l) {
    preconditions = [
      CAMION_EN(c, l)
      PAQUETE_EN(p, c)
    ]
    effects = [
      PAQUETE_EN(p, c) := false
      PAQUETE_EN(p, l) := true
    ]
  }


Antes de proseguir creando los elementos adicionales que se necesitan para especificar una instancia concreta del problema del transporte de paquetes, vamos a recopilar todos los elementos anteriores para poder escribir un fichero PDDL con la especificación del dominio del problema. Obérvese cómo al recopilar los predicados hay que indicar qué valor tienen por defecto, usándose en este caso el valor falso para implementar la hipótesis del mundo cerrado.

In [18]:
from unified_planning.shortcuts import Problem

In [19]:
problema_transporte_paquetes = Problem('Problema del transporte de paquetes')
for tipo_de_objeto in [paquete, localización, lugar, camión]:
    problema_transporte_paquetes.user_types.append(tipo_de_objeto)
for fluente in [conectados, camión_en, paquete_en]:
    problema_transporte_paquetes.add_fluent(fluente, default_initial_value=False)
problema_transporte_paquetes.add_actions([acción_ir, acción_cargar, acción_descargar])

In [20]:
print(problema_transporte_paquetes)

problem name = Problema del transporte de paquetes

types = [Paquete, Localizacion, Lugar - Localizacion, Camion - Localizacion]

fluents = [
  bool CONECTADOS[l1=Lugar - Localizacion, l2=Lugar - Localizacion]
  bool CAMION_EN[c=Camion - Localizacion, l=Lugar - Localizacion]
  bool PAQUETE_EN[p=Paquete, lc=Localizacion]
]

actions = [
  action IR(Camion - Localizacion c, Lugar - Localizacion l1, Lugar - Localizacion l2) {
    preconditions = [
      CONECTADOS(l1, l2)
      CAMION_EN(c, l1)
    ]
    effects = [
      CAMION_EN(c, l1) := false
      CAMION_EN(c, l2) := true
    ]
  }
  action CARGAR(Paquete p, Camion - Localizacion c, Lugar - Localizacion l) {
    preconditions = [
      CAMION_EN(c, l)
      PAQUETE_EN(p, l)
    ]
    effects = [
      PAQUETE_EN(p, l) := false
      PAQUETE_EN(p, c) := true
    ]
  }
  action DESCARGAR(Paquete p, Camion - Localizacion c, Lugar - Localizacion l) {
    preconditions = [
      CAMION_EN(c, l)
      PAQUETE_EN(p, c)
    ]
    effects = [

In [21]:
from unified_planning.io import PDDLWriter

In [22]:
escritor_PDDL = PDDLWriter(problema_transporte_paquetes)
escritor_PDDL.write_domain('dominio_transporte_paquetes.pddl')

Para completar la especificación de la instancia del problema de transporte de paquetes, hay que añadir en primer lugar los objetos concretos usados por esa instancia, indicando el tipo de cada uno de ellos.

In [23]:
from unified_planning.shortcuts import Object

In [24]:
lugares = [Object(f'L{i}', lugar) for i in range(4)]
C = Object('C', camión)
P = Object('P', paquete)
for objeto in lugares + [C, P]:
    print(objeto)
    problema_transporte_paquetes.add_object(objeto)

L0
L1
L2
L3
C
P


A continuación se establecen a verdadero los hechos del estado inicial.

In [25]:
for i, j in [(0, 1), (1, 2), (2, 3), (1, 3)]:
    Li = lugares[i]
    Lj = lugares[j]
    problema_transporte_paquetes.set_initial_value(conectados(Li, Lj), True)
    problema_transporte_paquetes.set_initial_value(conectados(Lj, Li), True)
problema_transporte_paquetes.set_initial_value(camión_en(C, lugares[0]), True)
problema_transporte_paquetes.set_initial_value(paquete_en(P, lugares[1]), True)

Finalmente, se indican los hechos objetivos.

In [26]:
problema_transporte_paquetes.add_goal(camión_en(C, lugares[0]))
problema_transporte_paquetes.add_goal(paquete_en(P, lugares[3]))

In [27]:
print(problema_transporte_paquetes)

problem name = Problema del transporte de paquetes

types = [Paquete, Localizacion, Lugar - Localizacion, Camion - Localizacion]

fluents = [
  bool CONECTADOS[l1=Lugar - Localizacion, l2=Lugar - Localizacion]
  bool CAMION_EN[c=Camion - Localizacion, l=Lugar - Localizacion]
  bool PAQUETE_EN[p=Paquete, lc=Localizacion]
]

actions = [
  action IR(Camion - Localizacion c, Lugar - Localizacion l1, Lugar - Localizacion l2) {
    preconditions = [
      CONECTADOS(l1, l2)
      CAMION_EN(c, l1)
    ]
    effects = [
      CAMION_EN(c, l1) := false
      CAMION_EN(c, l2) := true
    ]
  }
  action CARGAR(Paquete p, Camion - Localizacion c, Lugar - Localizacion l) {
    preconditions = [
      CAMION_EN(c, l)
      PAQUETE_EN(p, l)
    ]
    effects = [
      PAQUETE_EN(p, l) := false
      PAQUETE_EN(p, c) := true
    ]
  }
  action DESCARGAR(Paquete p, Camion - Localizacion c, Lugar - Localizacion l) {
    preconditions = [
      CAMION_EN(c, l)
      PAQUETE_EN(p, c)
    ]
    effects = [

In [28]:
escritor_PDDL = PDDLWriter(problema_transporte_paquetes)
escritor_PDDL.write_problem('problema_transporte_paquetes.pddl')

Una vez especificado el problema por completo, procedemos a buscar un plan solución con el planificador `Fast Downward`, indicándole que use para ello el algoritmo $\mathrm{A}^{*}$ y la heurística $h^{\mathrm{max}}$.

In [29]:
planificador = OneshotPlanner(name='fast-downward',
                              params={'fast_downward_search_config': 'astar(hmax())'})

  *** Credits ***
  * In operation mode `OneshotPlanner` at line 1 of `/tmp/ipykernel_28051/3851894197.py`, you are using the following planning engine:
  * Engine name: Fast Downward
  * Developers:  Uni Basel team and contributors (cf. https://github.com/aibasel/downward/blob/main/README.md)
  * Description: Fast Downward is a domain-independent classical planning system.



Aunque para especificar el problema del transporte de paquetes se ha usado una característica más avanzada que para el mundo de los bloques, la jerarquía de tipos, el planificador sigue siendo compatible con ella.

In [30]:
print(problema_transporte_paquetes.kind)

PROBLEM_CLASS: ['ACTION_BASED']
TYPING: ['FLAT_TYPING', 'HIERARCHICAL_TYPING']


In [31]:
print(planificador.supports(problema_transporte_paquetes.kind))

True


Como no hemos establecido un coste para las acciones, se asume que tienen coste unitario, por lo que un plan óptimo es un plan de longitud mínima.

In [32]:
resultado = planificador.solve(problema_transporte_paquetes)
print(resultado)

status: SOLVED_SATISFICING
engine: Fast Downward
plan: SequentialPlan:
    IR(C, L0, L1)
    CARGAR(P, C, L1)
    IR(C, L1, L3)
    DESCARGAR(P, C, L3)
    IR(C, L3, L1)
    IR(C, L1, L0)


Como la especificación del problema se ha realizado mediante esquemas de acciones, los costes de estas últimas dependerán en general de los argumentos del esquema correspondiente. Entonces la mejor manera de indicar esos costes es incluir en el problema un fluente numérico que dependa de los mismos argumentos. Por ejemplo, podríamos considerar que el coste de las acciones IR es la distancia entre el lugar de partida y el lugar de llegada, por lo que en primer lugar incluimos en el problema la información acerca de esas distancias.

In [33]:
from unified_planning.shortcuts import IntType, Int

distancia = Fluent('DISTANCIA', IntType(), l1=lugar, l2=lugar)
problema_transporte_paquetes.add_fluent(distancia, default_initial_value=Int(1))
problema_transporte_paquetes.set_initial_value(distancia(lugares[1], lugares[3]), Int(3))
problema_transporte_paquetes.set_initial_value(distancia(lugares[3], lugares[1]), Int(3))
print(problema_transporte_paquetes)

problem name = Problema del transporte de paquetes

types = [Paquete, Localizacion, Lugar - Localizacion, Camion - Localizacion]

fluents = [
  bool CONECTADOS[l1=Lugar - Localizacion, l2=Lugar - Localizacion]
  bool CAMION_EN[c=Camion - Localizacion, l=Lugar - Localizacion]
  bool PAQUETE_EN[p=Paquete, lc=Localizacion]
  integer DISTANCIA[l1=Lugar - Localizacion, l2=Lugar - Localizacion]
]

actions = [
  action IR(Camion - Localizacion c, Lugar - Localizacion l1, Lugar - Localizacion l2) {
    preconditions = [
      CONECTADOS(l1, l2)
      CAMION_EN(c, l1)
    ]
    effects = [
      CAMION_EN(c, l1) := false
      CAMION_EN(c, l2) := true
    ]
  }
  action CARGAR(Paquete p, Camion - Localizacion c, Lugar - Localizacion l) {
    preconditions = [
      CAMION_EN(c, l)
      PAQUETE_EN(p, l)
    ]
    effects = [
      PAQUETE_EN(p, l) := false
      PAQUETE_EN(p, c) := true
    ]
  }
  action DESCARGAR(Paquete p, Camion - Localizacion c, Lugar - Localizacion l) {
    preconditions 

A continuación creamos una métrica de minimización de coste, en la que asociamos esas distancias como coste de las acciones IR y al resto de acciones (es decir, a las acciones CARGAR y DESCARGAR) le asociamos coste 0.

In [34]:
from unified_planning.shortcuts import MinimizeActionCosts

métrica = MinimizeActionCosts({acción_ir: distancia(acción_ir.l1, acción_ir.l2)},
                              default=Int(0))
print(métrica)

minimize actions-cost: {'IR': DISTANCIA(l1, l2), 'default': 0}


Finalmente, añadimos la métrica anterior al problema para establecer que un plan solución debe ser un plan de coste mínimo.

In [35]:
problema_transporte_paquetes.add_quality_metric(métrica)
print(problema_transporte_paquetes)

problem name = Problema del transporte de paquetes

types = [Paquete, Localizacion, Lugar - Localizacion, Camion - Localizacion]

fluents = [
  bool CONECTADOS[l1=Lugar - Localizacion, l2=Lugar - Localizacion]
  bool CAMION_EN[c=Camion - Localizacion, l=Lugar - Localizacion]
  bool PAQUETE_EN[p=Paquete, lc=Localizacion]
  integer DISTANCIA[l1=Lugar - Localizacion, l2=Lugar - Localizacion]
]

actions = [
  action IR(Camion - Localizacion c, Lugar - Localizacion l1, Lugar - Localizacion l2) {
    preconditions = [
      CONECTADOS(l1, l2)
      CAMION_EN(c, l1)
    ]
    effects = [
      CAMION_EN(c, l1) := false
      CAMION_EN(c, l2) := true
    ]
  }
  action CARGAR(Paquete p, Camion - Localizacion c, Lugar - Localizacion l) {
    preconditions = [
      CAMION_EN(c, l)
      PAQUETE_EN(p, l)
    ]
    effects = [
      PAQUETE_EN(p, l) := false
      PAQUETE_EN(p, c) := true
    ]
  }
  action DESCARGAR(Paquete p, Camion - Localizacion c, Lugar - Localizacion l) {
    preconditions 

Buscamos ahora un plan óptimo usando el algoritmo $\mathrm{A}^{*}$ y, en este caso, la heurística $h^{\mathrm{add}}$.

In [36]:
planificador = OneshotPlanner(name='fast-downward',
                              params={'fast_downward_search_config': 'astar(add())'})

  *** Credits ***
  * In operation mode `OneshotPlanner` at line 1 of `/tmp/ipykernel_28051/405901140.py`, you are using the following planning engine:
  * Engine name: Fast Downward
  * Developers:  Uni Basel team and contributors (cf. https://github.com/aibasel/downward/blob/main/README.md)
  * Description: Fast Downward is a domain-independent classical planning system.



In [37]:
print(problema_transporte_paquetes.kind)

PROBLEM_CLASS: ['ACTION_BASED']
TYPING: ['FLAT_TYPING', 'HIERARCHICAL_TYPING']
QUALITY_METRICS: ['ACTIONS_COST']
ACTIONS_COST_KIND: ['STATIC_FLUENTS_IN_ACTIONS_COST', 'INT_NUMBERS_IN_ACTIONS_COST']


In [38]:
planificador.supports(problema_transporte_paquetes.kind)

True

In [39]:
resultado = planificador.solve(problema_transporte_paquetes)
print(resultado)

status: SOLVED_SATISFICING
engine: Fast Downward
plan: SequentialPlan:
    IR(C, L0, L1)
    CARGAR(P, C, L1)
    IR(C, L1, L2)
    IR(C, L2, L3)
    DESCARGAR(P, C, L3)
    IR(C, L3, L2)
    IR(C, L2, L1)
    IR(C, L1, L0)
